# 第 04 章练习

完成下面的练习，结合自己的运行结果作答。


<a id="s-04-06_chapter_test"></a>


## 4.6 ACL 调用流程

对照 GEMM 和 SpMV 的源码，比较数据准备、算子调用和资源释放。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


### 开始前

先完成 4.3、4.4 节的编译和运行。若尚未安装 ops-sparse，回到 4.4 节运行准备单元格，再继续本节。

### 设置库路径

使用 SpMV 小节安装的 ops-sparse。若安装在其他位置，修改下面的目录。


In [ ]:
import os

# 与安装步骤中的目录一致；自定义安装时修改这里。
os.environ["OPS_SPARSE_ROOT"] = os.path.expanduser("~/Ascend/cann")
os.environ["LD_LIBRARY_PATH"] = os.environ["OPS_SPARSE_ROOT"] + "/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")


### 编译并运行 GEMM


In [ ]:
%%bash
set -e
cd src/acl_operator_calls/GEMM-acl
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DACL_C_STUB=OFF
cmake --build build -j
# 学生修改：--m、--k、--n。
./build/bin/gemm_acl --m 1024 --k 1024 --n 1024 --warmup 3 --repeat 10


### 编译并运行 SpMV


In [ ]:
%%bash
set -e
cd src/acl_operator_calls/SpMV-acl
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DACL_C_STUB=OFF
cmake --build build -j
# 学生修改：--rows、--cols、--nnz。
./build/bin/spmv_acl --rows 100000 --cols 100000 --nnz 1000000 --warmup 3 --repeat 10


### 四类考核

对照 `src/acl_operator_calls/` 源码和上面的运行结果作答。

#### 1. 客观题

（1）单选：`SpMV-acl/src/main.cpp` 中执行 SpMV 的实际调用是（　）

A. 先查询 workspace 大小并用 `aclrtMalloc` 分配，再调用 `aclsparseSpMV`

B. 直接调用 `aclsparseSpMV(handle, ACL_SPARSE_OP_NON_TRANSPOSE, &alpha, matrix, vec_x, &beta, vec_y, ACL_FLOAT, ACL_SPARSE_SPMV_ALG_DEFAULT, nullptr)`，不查询、不分配 workspace

C. 先用额外 API 手动写入 CSR 数据，再调用 `aclsparseSpMV`

D. 把输出向量当作稠密矩阵，用稠密矩阵乘 API 计算

（2）判断（对/错）：SpMV 的释放顺序与创建顺序相反：`aclsparseDestroyDnVec`（vec_x、vec_y）→ `aclsparseDestroySpMat`（matrix）→ `aclsparseDestroy`（handle）。（　）

#### 2. 简单题

运行一次 GEMM 或 SpMV，找到输出中的相对误差，判断是否小于 `1e-6`。

#### 3. 中等题

分别运行 GEMM 和 SpMV，完成两项分析：

1. 根据 `copy_to_device`、算子调用和 `copy_to_host`，画出数据从主机到设备、再回到主机的过程。
2. 对照 `Timing scope`、误差和 `Workspace bytes` 等输出，解释两种 API 如何准备和释放资源。GEMM 使用 workspace/executor，SpMV 的 workspace 参数为 `nullptr`。

两者计算任务不同，本题比较调用流程。SpMV 没有单独输出 H2D/D2H 时间，分析时注明其计时范围。

#### 4. 困难题

使用不存在的设备编号做一次初始化失败实验。下面先用设备编号 9999 触发初始化错误，再用 Device 0 运行相同输入。

记录实际错误码、出错的 ACL 调用和已创建的资源，结合 Runtime 源码说明失败时怎样清理。检查 Device 0 复测的后端、设备编号和误差。此题只改运行参数，无需修改源码。


In [ ]:
%%bash
# 先观察不存在的设备编号导致的初始化失败。
./src/acl_operator_calls/GEMM-acl/build/bin/gemm_acl \
  --m 32 --n 32 --k 32 --warmup 0 --repeat 1 --device 9999
rc=$?
echo "失败实验退出码：$rc"
set -e
test "$rc" -ne 0
# 再用 Device 0 运行相同输入。
./src/acl_operator_calls/GEMM-acl/build/bin/gemm_acl \
  --m 32 --n 32 --k 32 --warmup 0 --repeat 1 --device 0


[参考答案](answer/04.06_answer.md)
